# pg-audit-analytics -- Интерактивный дашборд анализа аудиторских логов PostgreSQL

Данный дашборд визуализирует результаты анализа аудиторских логов PostgreSQL,
включая кластеризацию пользовательского поведения, обнаружение аномалий и анализ производительности запросов.

## Разделы:
1. Подключение к БД и загрузка данных
2. Топ-10 наиболее запрашиваемых таблиц
3. Тепловая карта операций по часам
4. Кластеризация пользователей
5. Временная шкала аномалий
6. Распределение производительности запросов
7. Разбивка активности по ролям

## 0. Импорт библиотек и настройка

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text
from sklearn.decomposition import PCA

# Добавляем корень проекта в sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from etl.config import get_connection_string, DB_CONFIG

# Настройка стилей графиков
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Библиотеки импортированы")

## 1. Подключение к БД и загрузка данных

In [ ]:
# Создаём SQLAlchemy engine
conn_str = get_connection_string()
engine = create_engine(conn_str)
print(f"Подключение к БД: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

In [ ]:
# Загружаем основные таблицы из схемы audit_data
print("Загрузка данных...")

audit_logs = pd.read_sql(
    "SELECT * FROM audit_data.audit_logs ORDER BY timestamp",
    engine
)
print(f"  audit_logs: {len(audit_logs)} записей")

user_activity = pd.read_sql(
    "SELECT * FROM audit_data.user_activity ORDER BY created_at",
    engine
)
print(f"  user_activity: {len(user_activity)} записей")

query_stats = pd.read_sql(
    "SELECT * FROM audit_data.query_stats ORDER BY execution_count DESC",
    engine
)
print(f"  query_stats: {len(query_stats)} записей")

try:
    clustering_results = pd.read_sql(
        "SELECT * FROM audit_data.clustering_results",
        engine
    )
    print(f"  clustering_results: {len(clustering_results)} записей")
except Exception:
    clustering_results = pd.DataFrame()
    print("  clustering_results: нет данных (запустите кластеризацию)")

try:
    anomaly_results = pd.read_sql(
        "SELECT * FROM audit_data.anomaly_results ORDER BY timestamp",
        engine
    )
    print(f"  anomaly_results: {len(anomaly_results)} записей")
except Exception:
    anomaly_results = pd.DataFrame()
    print("  anomaly_results: нет данных (запустите обнаружение аномалий)")

print(f"\nЗагружено {len(audit_logs)} записей аудиторских логов")

In [ ]:
# Преобразуем типы и смотрим общую информацию
audit_logs["timestamp"] = pd.to_datetime(audit_logs["timestamp"])
audit_logs["duration_ms"] = pd.to_numeric(audit_logs["duration_ms"], errors="coerce")

print("=" * 60)
print("ОБЗОР ДАННЫХ")
print("=" * 60)
print(f"Период данных: {audit_logs['timestamp'].min()} -- {audit_logs['timestamp'].max()}")
print(f"Уникальных пользователей: {audit_logs['username'].nunique()}")
print(f"Уникальных таблиц: {audit_logs['table_name'].nunique()}")
print(f"Типы операций: {audit_logs['operation_type'].unique()}")
print(f"Категории операций: {audit_logs['operation_category'].unique()}")
print(f"Пропуски в duration_ms: {audit_logs['duration_ms'].isna().sum()}")
print("=" * 60)

In [ ]:
# Первые 5 записей
audit_logs.head()

## 2. Топ-10 наиболее запрашиваемых таблиц (столбчатая диаграмма)

In [ ]:
# Считаем обращения к таблицам
table_counts = (
    audit_logs.dropna(subset=["table_name"])
    .groupby(["table_name", "operation_category"])
    .size()
    .reset_index(name="query_count")
)

# Топ-10 по общему количеству запросов
top_tables = (
    table_counts.groupby("table_name")["query_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_tables,
    x="table_name",
    y="query_count",
    title="Топ-10 наиболее запрашиваемых таблиц",
    labels={"table_name": "Таблица", "query_count": "Количество запросов"},
    color="query_count",
    color_continuous_scale="Blues",
)
fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white",
    height=500,
)
fig.show()

## 3. Тепловая карта операций по часам суток

In [ ]:
# Извлекаем час из timestamp
audit_logs["hour"] = audit_logs["timestamp"].dt.hour

# Группируем по часу и категории операций
heatmap_data = (
    audit_logs.groupby(["hour", "operation_category"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Тепловая карта операций по часам суток", fontsize=14, fontweight="bold")
ax.set_xlabel("Категория операции", fontsize=12)
ax.set_ylabel("Час суток", fontsize=12)
plt.tight_layout()

# Сохраняем график
os.makedirs(os.path.join(PROJECT_ROOT, "data", "processed"), exist_ok=True)
save_path = os.path.join(PROJECT_ROOT, "data", "processed", "heatmap_operations.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Тепловая карта сохранена: {save_path}")

plt.show()

## 4. Кластеризация пользователей

In [ ]:
from analytics.feature_eng import prepare_for_clustering
from analytics.clustering import run_clustering_analysis

print("Запуск кластеризации...")

# Подготавливаем признаки
X_scaled, user_ids, feature_names, scaler = prepare_for_clustering(audit_logs)
print(f"  Матрица признаков: {X_scaled.shape}")
print(f"  Пользователей: {len(user_ids)}")
print(f"  Признаков: {len(feature_names)}")

In [ ]:
# Запускаем полный анализ кластеризации
clustering = run_clustering_analysis(X_scaled, feature_names, user_ids)

km_labels = clustering["kmeans_labels"]
semantic_labels = clustering["semantic_labels"]
print(f"\nСемантические метки кластеров:")
for cid, label in semantic_labels.items():
    print(f"  Кластер {cid} -> {label}")

In [ ]:
# Визуализация кластеров в 2D через Plotly
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_.sum() * 100

cluster_df = pd.DataFrame({
    "username": user_ids,
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster_id": km_labels,
    "cluster_label": [semantic_labels.get(int(l), f"Кластер {l}") for l in km_labels],
})

fig = px.scatter(
    cluster_df,
    x="PC1",
    y="PC2",
    color="cluster_label",
    hover_data=["username"],
    title=f"Кластеризация пользователей (PCA -- {explained:.1f}% дисперсии)",
    labels={"cluster_label": "Кластер"},
    size_max=12,
)
fig.update_layout(
    template="plotly_white",
    height=600,
)
fig.show()

In [ ]:
# Профили кластеров -- средние значения признаков по каждому кластеру
feature_df = pd.DataFrame(X_scaled, columns=feature_names)
feature_df["username"] = user_ids
feature_df["cluster_label"] = cluster_df["cluster_label"]

cluster_profiles = feature_df.groupby("cluster_label").mean().reset_index()

# Преобразуем в длинный формат для графика
profile_long = cluster_profiles.melt(
    id_vars=["cluster_label"],
    var_name="feature",
    value_name="value",
)

fig = px.bar(
    profile_long,
    x="feature",
    y="value",
    color="cluster_label",
    barmode="group",
    title="Профили кластеров (стандартизированные признаки)",
    labels={"feature": "Признак", "value": "Значение", "cluster_label": "Кластер"},
)
fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white",
    height=500,
)
fig.show()

## 5. Временная шкала аномалий

In [ ]:
from analytics.anomaly_detection import run_anomaly_detection

print("Запуск обнаружения аномалий...")
anomalies = run_anomaly_detection(audit_logs, window="1h")

ts = anomalies["time_series"]
flags = anomalies["suspicious_flags"]
print(f"\nПодозрительных временных окон: {flags['is_suspicious'].sum()}")
print(f"Аномалий пользователей: {len(anomalies['user_anomalies'])}")

In [ ]:
# Интерактивная временная шкала транзакций с маркерами аномалий
anomaly_mask = flags["is_suspicious"]

fig = go.Figure()

# Линия транзакций
fig.add_trace(go.Scatter(
    x=ts.index,
    y=ts["transaction_count"],
    mode="lines",
    name="Транзакции",
    line=dict(color="steelblue", width=2),
    hovertemplate="Время: %{x}<br>Количество: %{y}<extra></extra>",
))

# Маркеры аномалий
anom_ts = ts[anomaly_mask]
if len(anom_ts) > 0:
    reasons = anom_ts.get("reasons", pd.Series(["Аномалия"] * len(anom_ts)))
    fig.add_trace(go.Scatter(
        x=anom_ts.index,
        y=anom_ts["transaction_count"],
        mode="markers",
        name="Аномалии",
        marker=dict(symbol="x", size=14, color="red", line=dict(width=2)),
        text=reasons,
        hovertemplate="Время: %{x}<br>Количество: %{y}<br>Причина: %{text}<extra></extra>",
    ))

fig.update_layout(
    title="Временная шкала транзакций с аномалиями",
    xaxis_title="Время",
    yaxis_title="Количество транзакций",
    hovermode="x unified",
    template="plotly_white",
    height=500,
)
fig.show()

In [ ]:
# Распределение аномалий по уровню серьёзности
time_anomalies = anomalies["time_anomalies"]

if time_anomalies:
    anom_df = pd.DataFrame(time_anomalies)
    anom_df["timestamp"] = pd.to_datetime(anom_df["timestamp"])
    anom_df = anom_df.sort_values("timestamp")

    severity_labels_ru = {"High": "Высокая", "Medium": "Средняя", "Low": "Низкая"}
    anom_df["severity_ru"] = anom_df["severity"].map(severity_labels_ru)

    fig = px.bar(
        anom_df,
        x="timestamp",
        y=anom_df["score"].abs(),
        color="severity_ru",
        color_discrete_map={"Высокая": "red", "Средняя": "orange", "Низкая": "green"},
        title="Распределение аномалий по уровню серьёзности",
        labels={"timestamp": "Время", "score": "Скор аномалии (абс.)", "severity_ru": "Серьёзность"},
        hover_data=["severity_ru", "score"],
    )
    fig.update_layout(
        template="plotly_white",
        height=450,
    )
    fig.show()
else:
    print("Аномалий не обнаружено")

In [ ]:
# Аномалии пользователей -- кто отклоняется от своего обычного поведения
user_anomalies = anomalies["user_anomalies"]

if user_anomalies:
    ua_df = pd.DataFrame(user_anomalies)
    ua_df["deviation_count"] = ua_df["deviations"].apply(len)
    ua_df = ua_df.sort_values("deviation_count", ascending=False)

    fig = px.bar(
        ua_df,
        x="username",
        y="deviation_count",
        title="Аномалии пользователей -- количество отклонений",
        labels={"username": "Пользователь", "deviation_count": "Количество отклонений"},
        color="deviation_count",
        color_continuous_scale="Reds",
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        template="plotly_white",
        height=450,
    )
    fig.show()

    print("\nДетали отклонений:")
    for _, row in ua_df.iterrows():
        print(f"  {row['username']}: {row['deviations']}")
else:
    print("Аномалий пользователей не обнаружено")

## 6. Распределение производительности запросов

In [ ]:
# Гистограмма времени выполнения запросов с перцентилями
df_with_duration = audit_logs.dropna(subset=["duration_ms"])

if len(df_with_duration) > 0:
    p50 = df_with_duration["duration_ms"].quantile(0.5)
    p95 = df_with_duration["duration_ms"].quantile(0.95)
    p99 = df_with_duration["duration_ms"].quantile(0.99)

    fig = px.histogram(
        df_with_duration,
        x="duration_ms",
        nbins=100,
        title="Распределение времени выполнения запросов",
        labels={"duration_ms": "Длительность (мс)"},
        color_discrete_sequence=["steelblue"],
    )
    fig.add_vline(x=p50, line_dash="dash", line_color="green", annotation_text=f"P50={p50:.0f}")
    fig.add_vline(x=p95, line_dash="dash", line_color="orange", annotation_text=f"P95={p95:.0f}")
    fig.add_vline(x=p99, line_dash="dash", line_color="red", annotation_text=f"P99={p99:.0f}")
    fig.update_layout(
        template="plotly_white",
        height=500,
    )
    fig.show()
else:
    print("Нет данных о длительности запросов")

In [ ]:
# Box plot длительности по типу операции
df_with_dur_and_op = audit_logs.dropna(subset=["duration_ms", "operation_category"])

if len(df_with_dur_and_op) > 0:
    fig = px.box(
        df_with_dur_and_op,
        x="operation_category",
        y="duration_ms",
        title="Длительность запросов по категории операций",
        labels={"operation_category": "Категория", "duration_ms": "Длительность (мс)"},
        color="operation_category",
        log_y=True,
    )
    fig.update_layout(
        template="plotly_white",
        height=500,
    )
    fig.show()

In [ ]:
# Если есть данные query_stats -- покажем медленные запросы
if len(query_stats) > 0:
    slow_queries = query_stats.nlargest(15, "avg_duration_ms")

    fig = px.bar(
        slow_queries,
        x="query_pattern",
        y="avg_duration_ms",
        title="Топ-15 самых медленных шаблонов запросов",
        labels={"query_pattern": "Шаблон запроса", "avg_duration_ms": "Средняя длительность (мс)"},
        color="avg_duration_ms",
        color_continuous_scale="Reds",
        hover_data=["execution_count", "p95_duration_ms", "max_duration_ms"],
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        template="plotly_white",
        height=550,
    )
    fig.show()

## 7. Разбивка активности по ролям

In [ ]:
# Круговая диаграмма -- распределение активности по пользователям
user_counts = audit_logs["username"].value_counts().reset_index()
user_counts.columns = ["username", "query_count"]

fig = px.pie(
    user_counts,
    values="query_count",
    names="username",
    title="Распределение активности по пользователям",
    hole=0.3,
)
fig.update_layout(
    template="plotly_white",
    height=500,
)
fig.show()

In [ ]:
# Стековая столбчатая диаграмма -- микс операций по пользователям
op_by_user = (
    audit_logs.groupby(["username", "operation_category"])
    .size()
    .unstack(fill_value=0)
)

# Нормализуем в проценты для наглядности
op_by_user_pct = op_by_user.div(op_by_user.sum(axis=1), axis=0) * 100
op_by_user_pct = op_by_user_pct.reset_index()

op_long = op_by_user_pct.melt(
    id_vars=["username"],
    var_name="operation_category",
    value_name="percentage",
)

fig = px.bar(
    op_long,
    x="username",
    y="percentage",
    color="operation_category",
    barmode="stack",
    title="Микс операций по пользователям (%)",
    labels={"username": "Пользователь", "percentage": "Доля (%)", "operation_category": "Категория"},
)
fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white",
    height=500,
)
fig.show()

In [ ]:
# Активность по дням недели
audit_logs["day_of_week"] = audit_logs["timestamp"].dt.day_name()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

day_counts = (
    audit_logs["day_of_week"]
    .value_counts()
    .reindex(day_order)
    .reset_index()
)
day_counts.columns = ["day_of_week", "query_count"]

day_names_ru = {
    "Monday": "Пн", "Tuesday": "Вт", "Wednesday": "Ср",
    "Thursday": "Чт", "Friday": "Пт", "Saturday": "Сб", "Sunday": "Вс",
}
day_counts["day_ru"] = day_counts["day_of_week"].map(day_names_ru)

fig = px.bar(
    day_counts,
    x="day_ru",
    y="query_count",
    title="Активность по дням недели",
    labels={"day_ru": "День недели", "query_count": "Количество запросов"},
    color="query_count",
    color_continuous_scale="Viridis",
)
fig.update_layout(
    template="plotly_white",
    height=450,
)
fig.show()

## 8. Итоговая сводка

In [ ]:
print("=" * 60)
print("ИТОГОВАЯ СВОДКА АНАЛИЗА АУДИТА")
print("=" * 60)
print(f"Всего запросов проанализировано: {len(audit_logs):,}")
print(f"Период: {audit_logs['timestamp'].min()} -- {audit_logs['timestamp'].max()}")
print(f"Уникальных пользователей: {audit_logs['username'].nunique()}")
print(f"Уникальных таблиц: {audit_logs['table_name'].nunique()}")
print(f"Среднее кол-во запросов на пользователя: {len(audit_logs) / audit_logs['username'].nunique():.1f}")

# Пиковый час
peak_hour = audit_logs["timestamp"].dt.hour.mode()
if len(peak_hour) > 0:
    print(f"Пиковый час: {int(peak_hour.iloc[0])}:00")

# Кластеры
n_clusters = len([v for v in semantic_labels.values() if v != "Noise/Outliers"])
print(f"Обнаружено кластеров: {n_clusters}")
for cid, label in semantic_labels.items():
    count = int(np.sum(km_labels == cid))
    print(f"  {label}: {count} пользователей")

# Аномалии
n_anomalies = flags["is_suspicious"].sum()
print(f"Временных окон с аномалиями: {n_anomalies}")
print(f"Пользователей с аномалиями: {len(user_anomalies)}")

# Длительность
if len(df_with_duration) > 0:
    print(f"Средняя длительность запроса: {df_with_duration['duration_ms'].mean():.1f} мс")
    print(f"P95 длительность: {p95:.1f} мс")
    print(f"P99 длительность: {p99:.1f} мс")

print("=" * 60)
print("Анализ завершён. Графики сохранены в data/processed/")
print("=" * 60)